# Prepare review data and MiniLM embeddings

This notebook cleans the reviews, creates the training, validation, and test groups, and turns the text into numbers for our difficulty model. You can run this notebook in VS Code instead of `scripts/prepare_review_embeddings.py`. You only need to use one of them.

Put `thecourseforum_all_reviews (1).csv` in the project's `data` folder. Open the project in VS Code, select the project's Python environment as the notebook kernel, and run the cells from top to bottom. No Colab upload or download is needed.

The final cell saves `data/embeddings_cleaned_new.npz`. It stops if that file already exists. To replace it, set `overwrite = True` in that cell, then rerun the difficulty model to update its results.

Some inspection cells display private review text. Clear all outputs before saving it to GitHub or sharing it outside the approved group. This notebook does not train LogisticIT or collect new reviews.

Install required packages

In [ ]:
# Install the packages into the python environment selected for this notebook.
%pip install -q sentence-transformers scikit-learn pandas numpy

Import the libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

Load the dataset

In [ ]:
# Find the project folder whether the notebook starts in scripts or the project root.
from pathlib import Path

current_folder = Path.cwd().resolve()
# Look in the current folder and its parents so this works from the project root or scripts folder.
project_root = next(
    (folder for folder in (current_folder, *current_folder.parents)
     if (folder / "scripts/prepare_review_embeddings.py").is_file()),
    None
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the project folder in VS Code.")

# Load the original reviews from the local data folder.
input_path = project_root / "data/thecourseforum_all_reviews (1).csv"
reviews = pd.read_csv(input_path)

print("Dataset shape:", reviews.shape)
print("\nColumns:")
print(reviews.columns.tolist())

Check the two columns

In [ ]:
# Check the required columns for missing values and look at the mix of ratings.
print("Missing review_text:", reviews["review_text"].isna().sum())
print("Missing difficulty_rating:", reviews["difficulty_rating"].isna().sum())

print("\nDifficulty rating distribution:")
print(reviews["difficulty_rating"].value_counts().sort_index())

print("\nFirst 3 reviews:")
print(reviews["review_text"].head(3))

Inspect the raw review structure

In [ ]:
# Look at a few raw reviews to see which webpage details were included.
for i in range(5):
    print(f"\n--- REVIEW {i} ---")
    print(reviews.loc[i, "review_text"])

Check the beginning and ending patterns across the dataset

In [ ]:
# Compare the starts and ends of reviews to check the cleaning rules.
print("FIRST 10 REVIEW TEXTS:")
print("=" * 80)

for i in range(10):
    text = reviews.loc[i, "review_text"]
    print(f"\nReview {i}:")
    print(text[:150])
    print("...")
    print(text[-100:])

Clean review text

In [ ]:
import re

def clean_review(text):
    # Remove the webpage heading containing the term, year, and average rating.
    # January is included so reviews from that term are cleaned too.
    text = re.sub(
        r'^(Fall|Spring|Summer|Winter|January)\s+\d{4}\s+\d+(?:\.\d+)?\s+AVERAGE\s*',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove the trailing vote count and posting date collected from the webpage.
    # Keep the student's actual writing between the heading and footer.
    text = re.sub(
        r'\s+-?\d+\s+'
        r'(?:Jan\.?|Feb\.?|Mar\.?|Apr\.?|May|Jun\.?|Jul\.?|Aug\.?|'
        r'Sept?\.?|Oct\.?|Nov\.?|Dec\.?|January|February|March|April|'
        r'June|July|August|September|October|November|December)'
        r'\s+\d{1,2},\s+\d{4}\s*$',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove extra spaces from the beginning and end of the cleaned review.
    return text.strip()

# Create a cleaned column while preserving the original review_text column.
reviews["cleaned_review"] = reviews["review_text"].apply(clean_review)

Check that the cleaning worked

In [ ]:
# Check a few cleaned reviews to see whether the extra webpage text was removed.
for i in range(5):
    print(f"\n--- CLEANED REVIEW {i} ---")
    print(reviews.loc[i, "cleaned_review"])

Create the features and target

In [ ]:
# Use the cleaned review text as the input.
X = reviews["cleaned_review"]

# Keep the actual difficulty ratings as the answers we want to predict.
y = reviews["difficulty_rating"]

# Check that every review still has a rating before making the split.
print("Number of reviews:", len(X))
print("Number of labels:", len(y))

print("\nTarget distribution:")
print(y.value_counts().sort_index())

Create the 70/15/15 split

In [ ]:
# Put 70 percent into training and keep a similar mix of ratings in each group.
# The seed of 42 repeats the split when the data and row order stay the same.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

print("Training set size:", len(X_train))
print("Temporary set size:", len(X_temp))

print("\nTraining rating distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTemporary rating distribution:")
print(y_temp.value_counts(normalize=True).sort_index())

Create validation and test sets

In [ ]:
# Divide the remaining reviews equally into validation and test groups.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training set size:", len(X_train))
print("Validation set size:", len(X_val))
print("Test set size:", len(X_test))

print("\nValidation rating distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTest rating distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Load MiniLM

In [ ]:
# Load the already trained MiniLM model. we are not training it again here.
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")

Generate training embeddings

In [ ]:
# Turn each training review into 384 numbers.
# Reviews stay together; text past the model's length limit is cut off.
X_train_embeddings = model.encode(
    X_train.tolist(),
    show_progress_bar=True
)

print("Training embeddings shape:", X_train_embeddings.shape)

Generate validation embeddings

In [ ]:
# Use the same model to turn the validation reviews into numbers.
X_val_embeddings = model.encode(
    X_val.tolist(),
    show_progress_bar=True
)

print("Validation embeddings shape:", X_val_embeddings.shape)

Generate test embeddings

In [ ]:
# Use the same model to turn the test reviews into numbers.
X_test_embeddings = model.encode(
    X_test.tolist(),
    show_progress_bar=True
)

print("Test embeddings shape:", X_test_embeddings.shape)

Final feature-engineering verification

In [ ]:
# Check that each group has 384 numbers per review and one rating per row.
print("FINAL DATA CHECK")
print("-" * 40)

print("Training:")
print("  Embeddings:", X_train_embeddings.shape)
print("  Labels:", y_train.shape)

print("\nValidation:")
print("  Embeddings:", X_val_embeddings.shape)
print("  Labels:", y_val.shape)

print("\nTest:")
print("  Embeddings:", X_test_embeddings.shape)
print("  Labels:", y_test.shape)

## Save embeddings to the data folder

To run the model with the existing embeddings, keep `overwrite = False` and skip this save cell. If you want to create the embeddings again from the raw CSV, run the preparation cells first, then set `overwrite = True` below to replace the existing file. Set it back to `False` afterward to avoid replacing it by accident on a later run.

If you want to keep the old embeddings too, change `output_path` below to a different filename ending in `.npz`. To use that new file, open `scripts/difficulty_model.py` and update this line to match its name and folder:

```python
input_path = root / "data/embeddings_cleaned_new.npz"
```

For example, if you save `data/embeddings_updated.npz`, change the model's line to `input_path = root / "data/embeddings_updated.npz"`. If you keep the same filename and folder, you do not need to change that line.

After replacing the embeddings or choosing a different file, rerun `difficulty_model.py` to update the scores and graphs. If the results change, update the written results in the READMEs and `output/model_results.md` too.

In [ ]:
# Save the review numbers and matching ratings where the difficulty model expects them.
output_path = project_root / "data/embeddings_cleaned_new.npz"
overwrite = False

# Only replace an existing file when we intentionally change this setting to true.
if output_path.exists() and not overwrite:
    raise FileExistsError("Embeddings already exist. Set overwrite = True only to replace them intentionally.")

# Keep the review numbers and ratings together in the same order for each group.
np.savez_compressed(
    output_path,
    X_train=X_train_embeddings,
    X_val=X_val_embeddings,
    X_test=X_test_embeddings,
    y_train=y_train.to_numpy(),
    y_val=y_val.to_numpy(),
    y_test=y_test.to_numpy()
)

print("Saved embeddings to:", output_path)